[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hanenalmayouf/applied-ml-workshop/blob/main/labs_colab/day2/lab_2_manafeth_preprocessing.ipynb)

# 🛠️ مختبر اليوم الثاني — تجهيز خصائص عملاء منافذ في خط معالجة

**ورشة أسس تعلم الآلة التطبيقي — اليوم 2 من 5**

هذا الدفتر مبني على مواصفات مختبرات «منافذ» المعتمدة للدورة، ومُجهَّز للعمل مباشرة في **Google Colab** أو في Jupyter محليًا.

**كيف تفتحه في Colab:** من قائمة `File → Upload notebook` في Colab ارفع هذا الملف. إذا لم يجد الدفتر مجلد البيانات تلقائيًا، ستظهر لك خانة لرفع ملف حزمة البيانات (`manafeth_data_package.zip`) المرفق بجوار هذا الدفتر — ارفعه وسيُستكمل التحميل تلقائيًا.

> 📌 راجع `00_start_here.md` قبل البدء لمعرفة طريقة استخدام خلايا **فكّر أولًا** و**TODO** و**مساعدة** في هذه الدفاتر.

In [ ]:
from pathlib import Path
import pandas as pd

# 1) نبحث عن مجلد البيانات بجانب هذا الدفتر (يعمل محليًا وفي Colab إذا رفعت المجلد كاملًا)
DATA_CANDIDATES = [Path("manafeth_data_package"), Path("data"), Path("../data/raw"), Path("data/raw")]
DATA_DIR = next((p for p in DATA_CANDIDATES if p.exists()), None)

# 2) إذا لم نجد المجلد ونحن داخل Google Colab، نطلب من الطالب رفع حزمة البيانات (ملف zip)
if DATA_DIR is None:
    try:
        from google.colab import files
        import zipfile

        print("لم يتم العثور على مجلد البيانات محليًا.")
        print("ارفع ملف حزمة البيانات (.zip) الذي يرافق هذا المختبر ثم انتظر انتهاء الرفع...")
        uploaded = files.upload()
        for name in uploaded:
            if name.lower().endswith(".zip"):
                with zipfile.ZipFile(name) as z:
                    z.extractall(".")
        DATA_DIR = next((p for p in DATA_CANDIDATES if p.exists()), Path("manafeth_data_package"))
    except ImportError:
        DATA_DIR = Path("manafeth_data_package")
        print("تنبيه: لسنا داخل Google Colab ولم يوجد مجلد بيانات — ضع حزمة البيانات بجانب الدفتر.")

CUSTOMERS_PATH = DATA_DIR / "manafeth_customers.parquet"
ORDERS_PATH = DATA_DIR / "manafeth_orders.parquet"
VEHICLES_PATH = DATA_DIR / "markabat_listings_sample.csv"
SHIFTED_PATH = DATA_DIR / "shifted_month.parquet"

print("مجلد البيانات المستخدم:", DATA_DIR.resolve())
assert CUSTOMERS_PATH.exists(), "تعذّر العثور على manafeth_customers.parquet — تأكد من رفع حزمة البيانات كاملة."

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.figsize"] = (7, 4)

## 🎯 هدف المختبر

تبني خط معالجة (Pipeline) يعالج النقص والخصائص العددية والفئوية بطريقة قابلة للتكرار، من دون أن يتعلم أي قيمة من بيانات الاختبار.

## السيناريو والبيانات

نستمر على جدول العملاء نفسه وعلى الخصائص الآمنة من اليوم الأول. المختبر يستفيد عمدًا من النقص الواقعي في `avg_rating` و`last_promo_used` — **لا تحذف هذين العمودين**؛ الهدف تعلّم معالجة النقص بصورة منظمة.

| مجموعة الخصائص | الأعمدة | المعالجة | السبب |
|---|---|---|---|
| عددية | `tenure_months`, `orders_per_month`, `avg_basket_sar`, `days_since_last_order`, `distinct_categories`, `promo_usage_rate`, `avg_rating` | وسيط ثم تحجيم + مؤشر نقص | `avg_rating` فيها نقص ذو معنى محتمل |
| فئوية | `city`, `device`, `payment_method`, `last_promo_used` | تعويض «غير معروف» ثم ترميز أحادي | الفئة اسم لا مقدار |
| ترتيبية | `city_tier` | تُعامل كرقم | الترتيب 1-2-3 موجود فعليًا |
| مستبعدة | `customer_id`, التواريخ, أعمدة التسريب, الهدف | لا تدخل `X` | معرّف أو معلومة مستقبلية |

## 🤔 فكّر أولًا

لماذا يجب حساب الوسيط (median) المستخدم لتعويض القيم الناقصة من بيانات **التدريب فقط**، لا من الجدول كاملًا قبل التقسيم؟

In [ ]:
customers = pd.read_parquet(CUSTOMERS_PATH)

safe_features = [
    "city", "city_tier", "device", "payment_method",
    "tenure_months", "orders_per_month", "avg_basket_sar",
    "days_since_last_order", "distinct_categories", "promo_usage_rate",
    "avg_rating", "last_promo_used"
]
X = customers[safe_features]
y = customers["churned_30d"]

In [ ]:
from sklearn.model_selection import train_test_split

# TODO: قسّم البيانات 80/20 مع الحفاظ على توازن الهدف (stratify) وrandom_state=42
X_train, X_test, y_train, y_test = train_test_split(...)
print(X_train.shape, X_test.shape)

### 📊 رسم: قبل التحجيم

مثال مُنفَّذ — التوزيع الخام لعمود `avg_basket_sar` قبل أي معالجة.

In [ ]:
X_train["avg_basket_sar"].plot(kind="hist", bins=30, color="#F26522")
plt.title("توزيع avg_basket_sar قبل التحجيم (خام)")
plt.xlabel("متوسط قيمة السلة (ريال)")
plt.show()

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

numeric_features = [
    "city_tier", "tenure_months", "orders_per_month", "avg_basket_sar",
    "days_since_last_order", "distinct_categories", "promo_usage_rate",
    "avg_rating"
]
categorical_features = ["city", "device", "payment_method", "last_promo_used"]

# TODO: ابنِ مسارًا عدديًا: تعويض بالوسيط مع مؤشر نقص (add_indicator=True)، ثم تحجيم StandardScaler
numeric_pipeline = Pipeline([
    ("fill", ...),
    ("scale", ...)
])

# TODO: ابنِ مسارًا فئويًا: تعويض بقيمة ثابتة "غير_معروف"، ثم OneHotEncoder(handle_unknown="ignore")
categorical_pipeline = Pipeline([
    ("fill", ...),
    ("encode", ...)
])

# TODO: اجمع المسارين داخل ColumnTransformer باسم preprocessor
preprocessor = ColumnTransformer([
    ("numbers", numeric_pipeline, numeric_features),
    ("categories", categorical_pipeline, categorical_features)
])

In [ ]:
prepared_train = preprocessor.fit_transform(X_train)
prepared_test = preprocessor.transform(X_test)
print(prepared_train.shape, prepared_test.shape)

### 📊 رسم: بعد التحجيم

قارن هذا بالمخطط الخام أعلاه — الشكل يتكرر لكن المحور الآن بمقياس موحّد (متوسط صفر وانحراف معياري 1).

In [ ]:
import numpy as np

# TODO: استخرج عمود avg_basket_sar المُحجَّم من prepared_train (تلميح: numeric_features.index(...))
col_index = ...
scaled_values = np.asarray(prepared_train)[:, col_index]

# TODO: ارسم مدرّج تكراري (histogram) لهذه القيم بعنوان مناسب

## النتيجة المتوقعة

ينتج متغير `preprocessor` ومصفوفتا خصائص مجهزتان للتدريب والاختبار، بعدد صفوف يطابق التدريب والاختبار. قد يزيد عدد الأعمدة بعد الترميز ومؤشر النقص؛ هذا طبيعي. لا يجب أن يضم أي ناتج عمودًا من أعمدة التسريب.

## المهارات التي راجعتها

فصل التدريب عن الاختبار، معالجة القيم الناقصة، استخدام مؤشر النقص، ترميز الفئات، تحجيم الأرقام، وبناء `Pipeline` و`ColumnTransformer`.

## ✅ تحقق ذاتيًا قبل إغلاق الدفتر
- [ ] `preprocessor` يعمل بدون أخطاء على `X_train` و`X_test`
- [ ] `prepared_train.shape[0]` يساوي عدد صفوف `X_train`
- [ ] تستطيع أن تشرح في جملة واحدة لماذا لا نحسب الوسيط من الجدول كاملًا قبل التقسيم

**غدًا:** نستخدم `preprocessor` نفسه لتدريب أول نموذج تصنيف وأول نموذج انحدار.